# TRIBE v2 quick test\n\nThis notebook runs Meta Research's official pretrained TRIBE v2 model on a short video. TRIBE v2 predicts an **average-subject** fMRI response on the fsaverage5 cortical mesh (~20k vertices). It does not measure your personal brain activity.\n\nOfficial repo: https://github.com/facebookresearch/tribev2

In [ ]:
!python -m pip install -U pip\n!pip install -q "torch>=2.5.1,<2.7" "torchvision>=0.20,<0.22" numpy==2.2.6 matplotlib pandas\n!pip install -q git+https://github.com/facebookresearch/tribev2.git

In [ ]:
from google.colab import files\n\nuploaded = files.upload()\nvideo_path = next(iter(uploaded))\nprint('Uploaded:', video_path)

In [ ]:
import numpy as np\nfrom tribev2 import TribeModel\n\nmodel = TribeModel.from_pretrained('facebook/tribev2', cache_folder='./cache')\nevents = model.get_events_dataframe(video_path=video_path)\npreds, segments = model.predict(events=events)\n\npreds = np.asarray(preds)\nprint('Prediction shape:', preds.shape)\nprint('Expected second dimension: ~20,000 cortical vertices')\nprint('Number of segments:', len(segments))

In [ ]:
import json\nimport numpy as np\n\nnp.save('predictions.npy', preds)\nwith open('segments.json', 'w') as f:\n    json.dump(segments, f, indent=2, default=str)\n\nsummary = {\n    'shape': list(preds.shape),\n    'mean': float(np.nanmean(preds)),\n    'std': float(np.nanstd(preds)),\n    'min': float(np.nanmin(preds)),\n    'max': float(np.nanmax(preds)),\n}\nprint(json.dumps(summary, indent=2))

In [ ]:
import matplotlib.pyplot as plt\n\nplt.figure(figsize=(12, 5))\nplt.imshow(preds[:, ::100].T, aspect='auto', interpolation='nearest')\nplt.xlabel('Prediction timestep')\nplt.ylabel('Cortical vertices (downsampled)')\nplt.title('TRIBE v2 predicted cortical activity')\nplt.colorbar(label='Predicted response')\nplt.tight_layout()

## Optional: 3D cortical visualization\n\nThe official package exposes a plotting extra. In a local clone of the TRIBE v2 package, install it with `pip install -e ".[plotting]"`, then follow the plotting helpers in the upstream demo notebook.\n\nThe model output is shifted by 5 seconds to compensate for hemodynamic lag.